### ベルヌーイ分布の成功確率の事後分布と事後統計量

In [1]:
import numpy as np
np.set_printoptions(precision=3)
import scipy as sp
import scipy.stats as st
import matplotlib.pyplot as plt
import japanize_matplotlib
%matplotlib inline
# import pymc
import psutil

#### 信用区間推定
##### 100(1-c)%信用区間
+ $ Pr\{ q < a_{c} | D \} = \frac{c}{2} $
+ $ Pr\{ q > b_{c} | D \} = \frac{c}{2} $

#### HPD(Heigest Posterior Density)区間推定
##### 100(1-c)%HPD区間
+ $ Pr\{a_{c} \leq q \leq b_{c} \} = 1 - c $
+ $ p(a_{c}|D) = p(b_{c}|D) $

In [2]:
import scipy.optimize as opt

def beta_hpdi(ci0, alpha, beta, prob) -> float:
    """ベータ分布のHPD区間の計算

    Args:
        ci0 (_type_): HPD区間の初期値
        alpha (_type_): ベータ分布のパラメータ1
        beta (_type_): ベータ分布のパラメータ2
        prob (_type_): HPD区間の確率 (0 < prob < 1)

    Returns:
        float, float: HPD区間
    """
    def hpdi_conditions(v, a, b, p) -> np.ndarray:
        """反復条件

        Args:
            v (_type_): HPD区間
            a (_type_): ベータ分布のパラメータ1
            b (_type_): ベータ分布のパラメータ2
            p (_type_): HPD区間の確率 (0 < prob < 1)

        Returns:
            np.ndarray: HPD区間の条件式の値
        """
        # 累積確率密度
        eq1 = st.beta.cdf(v[1], a, b) - st.beta.cdf(v[0], a, b) - p
        # 確率密度
        eq2 = st.beta.pdf(v[1], a, b) - st.beta.pdf(v[0], a, b)

        return np.hstack((eq1, eq2))
    
    return opt.root(hpdi_conditions, ci0, args=(alpha, beta, prob)).x